# song-search -> Geode port build (Windows x86_64)

# NOTE: the Geometry Dash version is NEVER hardcoded. It is discovered at runtime
# from the bindings clone and the SDK, then written into mod.json's gd.win.
Cross-compiles `matcool.song-search` for **Geometry Dash Windows (x86_64 / Win64)** on a
Linux kernel, per the Geode docs ("Required C++ Tools" -> Linux) plus matcool's own
`linux-win-cross` gist.

Heavy downloads (xwin splat, Geode SDK, bindings, toolchain) live under `/tmp/work`,
so the kernel output stays small: the mod source, the `.geode`, and `BUILD-LOG.txt`.

1. Install clang / cmake / git / ninja / lld
2. `xwin` splat the MSVC CRT + Windows SDK
3. `matcool/clang-msvc-sdk` CMake toolchain (clang-cl, `x86_64-pc-windows-msvc`)
4. Geode CLI + Geode SDK + Geode bindings (version discovered at runtime)
5. Write the mod source, run the pure-C++ logic test natively
6. Configure + build the `.geode`
7. Verify the built DLL architecture and the `mod.json`

CPU only, no GPU/TPU.

In [ ]:
%%bash

set -uo pipefail
mkdir -p /tmp/work
LOG=/kaggle/working/BUILD-LOG.txt
: > "$LOG"
log(){ echo "== $* ==" | tee -a "$LOG"; }

# --- deps as the Geode docs "Required C++ Tools -> Linux" describe ---
log "1. deps: clang clang-tools lld llvm cmake git"
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq >/dev/null 2>&1 || echo "apt update failed" >> "$LOG"
apt-get install -y -qq --no-install-recommends \
    git cmake ninja-build clang clang-tools lld llvm libssl3 \
    unzip xz-utils zip ca-certificates curl wget gpg >/dev/null 2>&1 \
  || echo "apt install partially failed" >> "$LOG"

# The docs note that some distros ship clang cmake that is too old: Geode
# needs Clang 19+ (21+ recommended) and CMake 3.29+, so pull both from the
# upstream repos when the default is too old.
. /etc/os-release
CODENAME=${VERSION_CODENAME:-noble}

CLANG_VER=$(clang --version | grep -oP 'version [0-9]+' | head -1 | grep -oP '[0-9]+')
if [ -z "$CLANG_VER" ] || [ "$CLANG_VER" -lt 19 ]; then
  log "clang ${CLANG_VER:-none} < 19 -> apt.llvm.org llvm-toolchain-$CODENAME-19"
  wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key 2>/dev/null \
    | gpg --dearmor -o /usr/share/keyrings/llvm.gpg || true
  echo "deb [signed-by=/usr/share/keyrings/llvm.gpg] https://apt.llvm.org/${CODENAME}/ llvm-toolchain-${CODENAME}-19 main" \
    > /etc/apt/sources.list.d/llvm19.list
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq --no-install-recommends clang-19 lld-19 llvm-19 \
    >/dev/null 2>&1 || echo "apt.llvm.org clang-19 failed" >> "$LOG"
  export PATH=/usr/lib/llvm-19/bin:$PATH
  ln -sf /usr/lib/llvm-19/bin/* /usr/local/bin/ 2>/dev/null || true
  hash -r
fi

CMAKE_VER=$(cmake --version | head -1 | grep -oP '[0-9]+\.[0-9]+' | head -1)
if [ -n "$CMAKE_VER" ] && awk "BEGIN{exit !($CMAKE_VER < 3.29)}"; then
  log "cmake $CMAKE_VER < 3.29 -> apt.kitware.com"
  wget -qO- https://apt.kitware.com/keys/kitware-archive-latest.asc 2>/dev/null \
    | gpg --dearmor -o /usr/share/keyrings/kitware.gpg || true
  echo "deb [signed-by=/usr/share/keyrings/kitware.gpg] https://apt.kitware.com/ubuntu ${CODENAME} main" \
    > /etc/apt/sources.list.d/kitware.list
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq cmake >/dev/null 2>&1 || true
  hash -r
fi

{
  echo "cmake : $(cmake --version | head -1)"
  echo "git   : $(git --version)"
  echo "ninja : $(ninja --version)"
  echo "clang : $(clang --version | head -1)"
  for t in clang-cl lld-link llvm-lib llvm-nm llvm-mt llvm-rc; do
    if command -v $t >/dev/null; then echo "$t : OK"; else echo "$t : MISSING"; fi
  done
} 2>&1 | tee -a "$LOG"

# Debian/Ubuntu never ship the clang-cl binary, and upstream dropped the
# x86_64 linux tarballs after 18.x. clang selects the CL driver purely from
# argv[0], so a link to clang named "clang-cl" is equivalent.
CLANG_DIR=$(dirname "$(command -v clang)")
if ! command -v clang-cl >/dev/null; then
  log "clang-cl absent -> linking $CLANG_DIR/clang as clang-cl"
  ln -sf "$CLANG_DIR/clang" "$CLANG_DIR/clang-cl" 2>/dev/null \
    || cp -f "$CLANG_DIR/clang" "$CLANG_DIR/clang-cl"
  ln -sf "$CLANG_DIR/clang" /usr/local/bin/clang-cl 2>/dev/null || true
  hash -r
fi

{
  echo "clang : $(clang --version | head -1)"
  echo "clang-cl: $(clang-cl --version 2>&1 | head -1)"
  for t in clang-cl lld-link llvm-lib llvm-nm llvm-mt llvm-rc; do
    if command -v $t >/dev/null; then echo "$t : OK"; else echo "$t : MISSING"; fi
  done
} 2>&1 | tee -a "$LOG"

command -v clang-cl >/dev/null || { echo "FATAL clang-cl missing" | tee -a "$LOG"; exit 2; }
command -v lld-link >/dev/null || { echo "FATAL lld-link missing" | tee -a "$LOG"; exit 2; }
log "DEPS DONE"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "2. xwin splat (MSVC CRT + Windows SDK)"

XWIN_VER=0.10.0
if [ ! -x /tmp/work/xwin/xwin ]; then
  curl -sSL -o /tmp/work/xwin.tar.gz \
    "https://github.com/Jake-Shadle/xwin/releases/download/${XWIN_VER}/xwin-${XWIN_VER}-x86_64-unknown-linux-musl.tar.gz" \
    || { echo "xwin download failed" | tee -a "$LOG"; exit 3; }
  mkdir -p /tmp/work/xwin
  tar -xzf /tmp/work/xwin.tar.gz -C /tmp/work/xwin --strip-components=1
  chmod +x /tmp/work/xwin/xwin
fi
export PATH=/tmp/work/xwin:$PATH
xwin --version | tee -a "$LOG"

# xwin's cache skips already-downloaded payloads, so retrying accumulates
# progress through a flaky connection. Default --arch is x86_64 and the
# default --variant is desktop; debug libs are excluded (Release build).
export XWIN_ACCEPT_LICENSE=1
DONE=0
for attempt in $(seq 1 60); do
  echo "--- xwin splat attempt $attempt/60 ---" >> "$LOG"
  # NB: --accept-license/--arch/--variant/--timeout/--cache-dir are GLOBAL
  # (before the subcommand); --output belongs to the splat subcommand.
  xwin --accept-license --arch x86_64 --variant desktop \
       --timeout 300 --cache-dir /tmp/work/xwin-cache \
       splat --include-debug-libs --output /tmp/work/splat >>"$LOG" 2>&1
  RC=$?
  if [ $RC -eq 0 ]; then
    DONE=1
    log "xwin splat OK on attempt $attempt"
    break
  fi
  echo "attempt $attempt rc=$RC" >> "$LOG"
  sleep 3
done
echo "xwin done=$DONE" | tee -a "$LOG"

echo "--- splat layout ---" | tee -a "$LOG"
du -sh /tmp/work/splat 2>>"$LOG" || true
for d in crt/include crt/lib/x86_64 \
         sdk/include/ucrt sdk/include/um sdk/include/shared sdk/include/winrt \
         sdk/lib/ucrt/x86_64 sdk/lib/um/x86_64; do
  if [ -d "/tmp/work/splat/$d" ]; then
    echo "  OK   $d" | tee -a "$LOG"
  else
    echo "  MISS $d" | tee -a "$LOG"
  fi
done
ls /tmp/work/splat/sdk/include/um/Windows.h >/dev/null 2>&1 \
  && echo "  Windows.h present" | tee -a "$LOG" \
  || echo "  Windows.h MISSING" | tee -a "$LOG"
[ "$DONE" = "1" ] || exit 4

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "3. toolchain + Geode CLI + Geode SDK + bindings"
mkdir -p /tmp/work && cd /tmp/work || exit 5
export PATH=/usr/local/bin:$PATH

if [ ! -d toolchain/.git ]; then
  git clone --depth 1 https://github.com/matcool/clang-msvc-sdk.git toolchain >>"$LOG" 2>&1 \
    || { echo "toolchain clone failed" | tee -a "$LOG"; exit 6; }
fi
ls toolchain | tee -a "$LOG"

GEODE_CLI_VER=v3.9.0
if [ ! -x /tmp/work/geode-cli/geode ]; then
  curl -sSL -o /tmp/work/geode.zip \
    "https://github.com/geode-sdk/cli/releases/download/${GEODE_CLI_VER}/geode-cli-${GEODE_CLI_VER}-linux.zip" \
    || { echo "geode cli download failed" | tee -a "$LOG"; exit 7; }
  rm -rf /tmp/work/geode-cli
  unzip -q -o /tmp/work/geode.zip -d /tmp/work/geode-cli
  chmod +x /tmp/work/geode-cli/geode
fi
export PATH=/tmp/work/geode-cli:$PATH
geode --version | tee -a "$LOG"

# `geode project check` (run by the SDK's CMake) needs a current CLI profile.
# Configure a throwaway one pointing at a scratch dir - we only cross-build
# here, never run the game. It also wants the usual Geode install layout
# present so it can read the list of installed mods.
mkdir -p /tmp/work/gd-scratch/geode/mods
mkdir -p /tmp/work/gd-scratch/geode/config
mkdir -p /tmp/work/gd-scratch/geode/logs
mkdir -p /tmp/work/gd-scratch/geode/unzipped
if ! geode profile list 2>/dev/null | grep -q kaggle; then
  geode profile add --name kaggle /tmp/work/gd-scratch >>"$LOG" 2>&1 || true
fi
geode profile switch kaggle >>"$LOG" 2>&1 || true
echo "profiles:" | tee -a "$LOG"
geode profile list 2>&1 | tee -a "$LOG"

if [ ! -d sdk/.git ]; then
  git clone --depth 1 --branch main https://github.com/geode-sdk/sdk.git sdk >>"$LOG" 2>&1 \
    || { echo "geode sdk clone failed" | tee -a "$LOG"; exit 8; }
fi
geode sdk set-path "/tmp/work/sdk" >>"$LOG" 2>&1 || true
geode sdk version | tee -a "$LOG"
ls sdk/loader/include/link/win64 | tee -a "$LOG"

# The SDK links the Geode loader import library from <sdk>/bin/<version>/.
# `geode sdk install-binaries` fetches it; do it manually if the CLI cannot.
SDK_VER=$(cat sdk/VERSION 2>/dev/null | tr -d '[:space:]')
echo "sdk version: $SDK_VER" | tee -a "$LOG"
if [ -n "$SDK_VER" ] && [ ! -f "sdk/bin/${SDK_VER}/Geode.lib" ]; then
  geode sdk install-binaries >>"$LOG" 2>&1 || true
fi
if [ ! -f "sdk/bin/${SDK_VER}/Geode.lib" ]; then
  log "install-binaries did not provide Geode.lib -> fetching loader zip directly"
  for attempt in $(seq 1 20); do
    curl -sSL -o /tmp/work/geode-win.zip \
      "https://github.com/geode-sdk/geode/releases/download/v${SDK_VER}/geode-v${SDK_VER}-win.zip" && break
    echo "loader zip attempt $attempt failed" >> "$LOG"
    sleep 5
  done
  rm -rf /tmp/work/geode-win && mkdir -p /tmp/work/geode-win
  unzip -q -o /tmp/work/geode-win.zip -d /tmp/work/geode-win || true
  mkdir -p "sdk/bin/${SDK_VER}"
  FOUND=$(find /tmp/work/geode-win -name 'Geode.lib' | head -1)
  if [ -n "$FOUND" ]; then
    cp -f "$FOUND" "sdk/bin/${SDK_VER}/Geode.lib"
    log "installed Geode.lib from $FOUND"
  else
    echo "WARNING: Geode.lib not found in loader zip" | tee -a "$LOG"
    find /tmp/work/geode-win -maxdepth 3 -type f | head -30 | tee -a "$LOG"
  fi
fi
ls -la "sdk/bin/${SDK_VER}/" 2>&1 | tee -a "$LOG"

# Geode generates binding headers at configure time from the bindings repo;
# pre-seed a clone so configure does not fetch it itself
if [ ! -d bindings/.git ]; then
  git clone --depth 1 https://github.com/geode-sdk/bindings.git bindings >>"$LOG" 2>&1 \
    || { echo "bindings clone failed" | tee -a "$LOG"; exit 9; }
fi
# ---- discover the Geometry Dash version we are compiling against ----
# NEVER hardcode this. Geode keeps publishing new bindings, and the SDK
# FATAL_ERRORs when mod.json's gd.win disagrees with the bindings version
# it was built against (GeodeFile.cmake compares gd.<platform> to
# GEODE_GD_VERSION). So: read the version out of the clone we just made.
sdk_gd_win() {
  sed -n '/# begin GEODE_GD_VERSION/,/# end GEODE_GD_VERSION/p' sdk/CMakeLists.txt \
    | sed -n 's/^#[[:space:]]*win:[[:space:]]*//p' | head -1 | tr -d '[:space:]'
}
# primary: the highest version directory that actually exists in the clone
GD_WIN=$(ls bindings/bindings 2>/dev/null \
  | grep -E '^[0-9]+(\.[0-9]+)+$' | sort -V | tail -1)
# fallback: the version the SDK itself was released against
SDK_GD_WIN=$(sdk_gd_win)
if [ -z "$GD_WIN" ]; then GD_WIN=$SDK_GD_WIN; fi
[ -n "$GD_WIN" ] || { echo "FATAL: could not discover the GD version" | tee -a "$LOG"; exit 9; }
if [ -n "$SDK_GD_WIN" ] && [ "$SDK_GD_WIN" != "$GD_WIN" ]; then
  echo "WARNING: SDK targets GD $SDK_GD_WIN but bindings dir listing picked $GD_WIN" | tee -a "$LOG"
fi
printf '%s\n' "$GD_WIN" > /tmp/work/gd-win.txt
echo "bindings GD version (win): $GD_WIN" | tee -a "$LOG"
ls "bindings/bindings/$GD_WIN" | tee -a "$LOG"
log "TOOLCHAIN DONE"

In [ ]:
import base64, json, pathlib
FILES = {
 "mod.json": "ewoJImdlb2RlIjogIjUuMTEuMSIsCgkiZ2QiOiB7CgkJIndpbiI6ICIyLjIwODEiLAoJCSJhbmRyb2lkIjogIioiLAoJCSJtYWMiOiAiKiIsCgkJImlvcyI6ICIqIgoJfSwKCSJpZCI6ICJtYXRjb29sLnNvbmctc2VhcmNoIiwKCSJuYW1lIjogIlNvbmcgU2VhcmNoIiwKCSJ2ZXJzaW9uIjogInYxLjAuMCIsCgkiZGV2ZWxvcGVyIjogIm1hdGNvb2wiLAoJImRlc2NyaXB0aW9uIjogIlNlYXJjaCBkb3dubG9hZGVkIHNvbmdzIHdpdGggZnV6enkgbWF0Y2hpbmcuIFBvcnQgb2YgbWF0Y29vbC9zbWFsbC1nZC1tb2RzIHNvbmctc2VhcmNoIHRvIEdlb2RlLiIKfQo=",
 "CMakeLists.txt": "Y21ha2VfbWluaW11bV9yZXF1aXJlZChWRVJTSU9OIDMuMjEpCnNldChDTUFLRV9DWFhfU1RBTkRBUkQgMjMpCnNldChDTUFLRV9DWFhfU1RBTkRBUkRfUkVRVUlSRUQgT04pCnNldChDTUFLRV9DWFhfVklTSUJJTElUWV9QUkVTRVQgaGlkZGVuKQoKcHJvamVjdChzb25nLXNlYXJjaCBWRVJTSU9OIDEuMC4wKQoKIyBBZGQgYWxsIHNvdXJjZSBmaWxlcyBpbnNpZGUgc3JjIChyZWN1cnNpdmVseSkKZmlsZShHTE9CX1JFQ1VSU0UgU09VUkNFUyBDT05GSUdVUkVfREVQRU5EUyBzcmMvKi5jcHApCgojIFNldCB1cCB0aGUgbW9kIGJpbmFyeQphZGRfbGlicmFyeSgke1BST0pFQ1RfTkFNRX0gU0hBUkVEICR7U09VUkNFU30pCgppZiAoTk9UIERFRklORUQgRU5We0dFT0RFX1NES30pCiAgICBtZXNzYWdlKEZBVEFMX0VSUk9SICJVbmFibGUgdG8gZmluZCBHZW9kZSBTREshIFBsZWFzZSBkZWZpbmUgR0VPREVfU0RLIGVudmlyb25tZW50IHZhcmlhYmxlIHRvIHBvaW50IHRvIEdlb2RlIikKZWxzZSgpCiAgICBtZXNzYWdlKFNUQVRVUyAiRm91bmQgR2VvZGU6ICRFTlZ7R0VPREVfU0RLfSIpCmVuZGlmKCkKCmFkZF9zdWJkaXJlY3RvcnkoJEVOVntHRU9ERV9TREt9ICR7Q01BS0VfQ1VSUkVOVF9CSU5BUllfRElSfS9nZW9kZSkKCiMgU2V0IHVwIGRlcGVuZGVuY2llcywgcmVzb3VyY2VzLCBhbmQgbGluayBHZW9kZS4Kc2V0dXBfZ2VvZGVfbW9kKCR7UFJPSkVDVF9OQU1FfSkK",
 ".gitignore": "YnVpbGQvCi50b29scy8KKi5nZW9kZQo=",
 "src/main.cpp": "...",
 "src/fts_fuzzy_match.h": "Ly8gTElDRU5TRQovLwovLyAgIFRoaXMgc29mdHdhcmUgaXMgZHVhbC1saWNlbnNlZCB0byB0aGUgcHVibGljIGRvbWFpbiBhbmQgdW5kZXIgdGhlIGZvbGxvd2luZwovLyAgIGxpY2Vuc2U6IHlvdSBhcmUgZ3JhbnRlZCBhIHBlcnBldHVhbCwgaXJyZXZvY2FibGUgbGljZW5zZSB0byBjb3B5LCBtb2RpZnksCi8vICAgcHVibGlzaCwgYW5kIGRpc3RyaWJ1dGUgdGhpcyBmaWxlIGFzIHlvdSBzZWUgZml0LgovLwovLyBWRVJTSU9OCi8vICAgMC4yLjAgICgyMDE3LTAyLTE4KSAgU2NvcmVkIG1hdGNoZXMgcGVyZm9ybSBleGhhdXN0aXZlIHNlYXJjaCBmb3IgYmVzdCBzY29yZQovLyAgIDAuMS4wICAoMjAxNi0wMy0yOCkgIEluaXRpYWwgcmVsZWFzZQovLwovLyBBVVRIT1IKLy8gICBGb3JyZXN0IFNtaXRoCi8vCi8vIE5PVEVTCi8vICAgQ29tcGlsaW5nCi8vICAgICBZb3UgTVVTVCBhZGQgJyNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OJyBiZWZvcmUgaW5jbHVkaW5nIHRoaXMgaGVhZGVyIGluIE9ORSBzb3VyY2UgZmlsZSB0byBjcmVhdGUgaW1wbGVtZW50YXRpb24uCi8vCi8vICAgZnV6enlfbWF0Y2hfc2ltcGxlKC4uLikKLy8gICAgIFJldHVybnMgdHJ1ZSBpZiBlYWNoIGNoYXJhY3RlciBpbiBwYXR0ZXJuIGlzIGZvdW5kIHNlcXVlbnRpYWxseSB3aXRoaW4gc3RyCi8vCi8vICAgZnV6enlfbWF0Y2goLi4uKQovLyAgICAgUmV0dXJucyB0cnVlIGlmIHBhdHRlcm4gaXMgZm91bmQgQU5EIGNhbGN1bGF0ZXMgYSBzY29yZS4KLy8gICAgIFBlcmZvcm1zIGV4aGF1c3RpdmUgc2VhcmNoIHZpYSByZWN1cnNpb24gdG8gZmluZCBhbGwgcG9zc2libGUgbWF0Y2hlcyBhbmQgbWF0Y2ggd2l0aCBoaWdoZXN0IHNjb3JlLgovLyAgICAgU2NvcmVzIHZhbHVlcyBoYXZlIG5vIGludHJpbnNpYyBtZWFuaW5nLiBQb3NzaWJsZSBzY29yZSByYW5nZSBpcyBub3Qgbm9ybWFsaXplZCBhbmQgdmFyaWVzIHdpdGggcGF0dGVybi4KLy8gICAgIFJlY3Vyc2lvbiBpcyBsaW1pdGVkIGludGVybmFsbHkgKGRlZmF1bHQ9MTApIHRvIHByZXZlbnQgZGVnZW5lcmF0ZSBjYXNlcyAocGF0dGVybj0iYWFhYWFhIiBzdHI9ImFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYSIpCi8vICAgICBVc2VzIHVpbnQ4X3QgZm9yIG1hdGNoIGluZGljZXMuIFRoZXJlZm9yZSBwYXR0ZXJucyBhcmUgbGltaXRlZCB0byAyNTYgY2hhcmFjdGVycy4KLy8gICAgIFNjb3JlIHN5c3RlbSBzaG91bGQgYmUgdHVuZWQgZm9yIFlPVVIgdXNlIGNhc2UuIFdvcmRzLCBzZW50ZW5jZXMsIGZpbGUgbmFtZXMsIG9yIG1ldGhvZCBuYW1lcyBhbGwgcHJlZmVyIGRpZmZlcmVudCB0dW5pbmcuCgojaWZuZGVmIEZUU19GVVpaWV9NQVRDSF9ICiNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0gKCiNpbmNsdWRlIDxjc3RkaW50PiAvLyB1aW50OF90CiNpbmNsdWRlIDxjdHlwZS5oPiAvLyA6OnRvbG93ZXIsIDo6dG91cHBlcgojaW5jbHVkZSA8Y3N0cmluZz4gLy8gbWVtY3B5CgojaW5jbHVkZSA8Y3N0ZGlvPgoKLy8gUHVibGljIGludGVyZmFjZQpuYW1lc3BhY2UgZnRzIHsKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cik7CiAgICBzdGF0aWMgYm9vbCBmdXp6eV9tYXRjaChjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0ciwgaW50ICYgb3V0U2NvcmUpOwogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlLCB1aW50OF90ICogbWF0Y2hlcywgaW50IG1heE1hdGNoZXMpOwp9CgojaWZkZWYgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OCm5hbWVzcGFjZSBmdHMgewoKICAgIC8vIEZvcndhcmQgZGVjbGFyYXRpb25zIGZvciAicHJpdmF0ZSIgaW1wbGVtZW50YXRpb24KICAgIG5hbWVzcGFjZSBmdXp6eV9pbnRlcm5hbCB7CiAgICAgICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2hfcmVjdXJzaXZlKGNvbnN0IGNoYXIgKiBwYXR0ZXJuLCBjb25zdCBjaGFyICogc3RyLCBpbnQgJiBvdXRTY29yZSwgY29uc3QgY2hhciAqIHN0ckJlZ2luLAogICAgICAgICAgICB1aW50OF90IGNvbnN0ICogc3JjTWF0Y2hlcywgIHVpbnQ4X3QgKiBuZXdNYXRjaGVzLCAgaW50IG1heE1hdGNoZXMsIGludCBuZXh0TWF0Y2gsCiAgICAgICAgICAgIGludCAmIHJlY3Vyc2lvbkNvdW50LCBpbnQgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFB1YmxpYyBpbnRlcmZhY2UKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cikgewogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgIHsKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpCiAgICAgICAgICAgICAgICArK3BhdHRlcm47CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgcmV0dXJuICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CiAgICB9CgogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlKSB7CgogICAgICAgIHVpbnQ4X3QgbWF0Y2hlc1syNTZdOwogICAgICAgIHJldHVybiBmdXp6eV9tYXRjaChwYXR0ZXJuLCBzdHIsIG91dFNjb3JlLCBtYXRjaGVzLCBzaXplb2YobWF0Y2hlcykpOwogICAgfQoKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoKGNoYXIgY29uc3QgKiBwYXR0ZXJuLCBjaGFyIGNvbnN0ICogc3RyLCBpbnQgJiBvdXRTY29yZSwgdWludDhfdCAqIG1hdGNoZXMsIGludCBtYXhNYXRjaGVzKSB7CiAgICAgICAgaW50IHJlY3Vyc2lvbkNvdW50ID0gMDsKICAgICAgICBpbnQgcmVjdXJzaW9uTGltaXQgPSAxMDsKCiAgICAgICAgcmV0dXJuIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyLCBvdXRTY29yZSwgc3RyLCBudWxscHRyLCBtYXRjaGVzLCBtYXhNYXRjaGVzLCAwLCByZWN1cnNpb25Db3VudCwgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFByaXZhdGUgaW1wbGVtZW50YXRpb24KICAgIHN0YXRpYyBib29sIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUoY29uc3QgY2hhciAqIHBhdHRlcm4sIGNvbnN0IGNoYXIgKiBzdHIsIGludCAmIG91dFNjb3JlLAogICAgICAgIGNvbnN0IGNoYXIgKiBzdHJCZWdpbiwgdWludDhfdCBjb25zdCAqIHNyY01hdGNoZXMsIHVpbnQ4X3QgKiBtYXRjaGVzLCBpbnQgbWF4TWF0Y2hlcywKICAgICAgICBpbnQgbmV4dE1hdGNoLCBpbnQgJiByZWN1cnNpb25Db3VudCwgaW50IHJlY3Vyc2lvbkxpbWl0KQogICAgewogICAgICAgIC8vIENvdW50IHJlY3Vyc2lvbnMKICAgICAgICArK3JlY3Vyc2lvbkNvdW50OwogICAgICAgIGlmIChyZWN1cnNpb25Db3VudCA+PSByZWN1cnNpb25MaW1pdCkKICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAvLyBEZXRlY3QgZW5kIG9mIHN0cmluZ3MKICAgICAgICBpZiAoKnBhdHRlcm4gPT0gJ1wwJyB8fCAqc3RyID09ICdcMCcpCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKCiAgICAgICAgLy8gUmVjdXJzaW9uIHBhcmFtcwogICAgICAgIGJvb2wgcmVjdXJzaXZlTWF0Y2ggPSBmYWxzZTsKICAgICAgICB1aW50OF90IGJlc3RSZWN1cnNpdmVNYXRjaGVzWzI1Nl07CiAgICAgICAgaW50IGJlc3RSZWN1cnNpdmVTY29yZSA9IDA7CgogICAgICAgIC8vIExvb3AgdGhyb3VnaCBwYXR0ZXJuIGFuZCBzdHIgbG9va2luZyBmb3IgYSBtYXRjaAogICAgICAgIGJvb2wgZmlyc3RfbWF0Y2ggPSB0cnVlOwogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgewoKICAgICAgICAgICAgLy8gRm91bmQgbWF0Y2gKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpIHsKCiAgICAgICAgICAgICAgICAvLyBTdXBwbGllZCBtYXRjaGVzIGJ1ZmZlciB3YXMgdG9vIHNob3J0CiAgICAgICAgICAgICAgICBpZiAobmV4dE1hdGNoID49IG1heE1hdGNoZXMpCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAgICAgICAgIC8vICJDb3B5LW9uLVdyaXRlIiBzcmNNYXRjaGVzIGludG8gbWF0Y2hlcwogICAgICAgICAgICAgICAgaWYgKGZpcnN0X21hdGNoICYmIHNyY01hdGNoZXMpIHsKICAgICAgICAgICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgc3JjTWF0Y2hlcywgbmV4dE1hdGNoKTsKICAgICAgICAgICAgICAgICAgICBmaXJzdF9tYXRjaCA9IGZhbHNlOwogICAgICAgICAgICAgICAgfQoKICAgICAgICAgICAgICAgIC8vIFJlY3Vyc2l2ZSBjYWxsIHRoYXQgInNraXBzIiB0aGlzIG1hdGNoCiAgICAgICAgICAgICAgICB1aW50OF90IHJlY3Vyc2l2ZU1hdGNoZXNbMjU2XTsKICAgICAgICAgICAgICAgIGludCByZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgICAgIGlmIChmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyICsgMSwgcmVjdXJzaXZlU2NvcmUsIHN0ckJlZ2luLCBtYXRjaGVzLCByZWN1cnNpdmVNYXRjaGVzLCBzaXplb2YocmVjdXJzaXZlTWF0Y2hlcyksIG5leHRNYXRjaCwgcmVjdXJzaW9uQ291bnQsIHJlY3Vyc2lvbkxpbWl0KSkgewoKICAgICAgICAgICAgICAgICAgICAvLyBQaWNrIGJlc3QgcmVjdXJzaXZlIHNjb3JlCiAgICAgICAgICAgICAgICAgICAgaWYgKCFyZWN1cnNpdmVNYXRjaCB8fCByZWN1cnNpdmVTY29yZSA+IGJlc3RSZWN1cnNpdmVTY29yZSkgewogICAgICAgICAgICAgICAgICAgICAgICBtZW1jcHkoYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIHJlY3Vyc2l2ZU1hdGNoZXMsIDI1Nik7CiAgICAgICAgICAgICAgICAgICAgICAgIGJlc3RSZWN1cnNpdmVTY29yZSA9IHJlY3Vyc2l2ZVNjb3JlOwogICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICAgICByZWN1cnNpdmVNYXRjaCA9IHRydWU7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQWR2YW5jZQogICAgICAgICAgICAgICAgbWF0Y2hlc1tuZXh0TWF0Y2grK10gPSAodWludDhfdCkoc3RyIC0gc3RyQmVnaW4pOwogICAgICAgICAgICAgICAgKytwYXR0ZXJuOwogICAgICAgICAgICB9CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgLy8gRGV0ZXJtaW5lIGlmIGZ1bGwgcGF0dGVybiB3YXMgbWF0Y2hlZAogICAgICAgIGJvb2wgbWF0Y2hlZCA9ICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CgogICAgICAgIC8vIENhbGN1bGF0ZSBzY29yZQogICAgICAgIGlmIChtYXRjaGVkKSB7CiAgICAgICAgICAgIGNvbnN0IGludCBzZXF1ZW50aWFsX2JvbnVzID0gMTU7ICAgICAgICAgICAgLy8gYm9udXMgZm9yIGFkamFjZW50IG1hdGNoZXMKICAgICAgICAgICAgY29uc3QgaW50IHNlcGFyYXRvcl9ib251cyA9IDMwOyAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBvY2N1cnMgYWZ0ZXIgYSBzZXBhcmF0b3IKICAgICAgICAgICAgY29uc3QgaW50IGNhbWVsX2JvbnVzID0gMzA7ICAgICAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBpcyB1cHBlcmNhc2UgYW5kIHByZXYgaXMgbG93ZXIKICAgICAgICAgICAgY29uc3QgaW50IGZpcnN0X2xldHRlcl9ib251cyA9IDE1OyAgICAgICAgICAvLyBib251cyBpZiB0aGUgZmlyc3QgbGV0dGVyIGlzIG1hdGNoZWQKCiAgICAgICAgICAgIGNvbnN0IGludCBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTU7ICAgICAgLy8gcGVuYWx0eSBhcHBsaWVkIGZvciBldmVyeSBsZXR0ZXIgaW4gc3RyIGJlZm9yZSB0aGUgZmlyc3QgbWF0Y2gKICAgICAgICAgICAgY29uc3QgaW50IG1heF9sZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTE1OyAvLyBtYXhpbXVtIHBlbmFsdHkgZm9yIGxlYWRpbmcgbGV0dGVycwogICAgICAgICAgICBjb25zdCBpbnQgdW5tYXRjaGVkX2xldHRlcl9wZW5hbHR5ID0gLTE7ICAgIC8vIHBlbmFsdHkgZm9yIGV2ZXJ5IGxldHRlciB0aGF0IGRvZXNuJ3QgbWF0dGVyCgogICAgICAgICAgICAvLyBJdGVyYXRlIHN0ciB0byBlbmQKICAgICAgICAgICAgd2hpbGUgKCpzdHIgIT0gJ1wwJykKICAgICAgICAgICAgICAgICsrc3RyOwoKICAgICAgICAgICAgLy8gSW5pdGlhbGl6ZSBzY29yZQogICAgICAgICAgICBvdXRTY29yZSA9IDEwMDsKCiAgICAgICAgICAgIC8vIEFwcGx5IGxlYWRpbmcgbGV0dGVyIHBlbmFsdHkKICAgICAgICAgICAgaW50IHBlbmFsdHkgPSBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ICogbWF0Y2hlc1swXTsKICAgICAgICAgICAgaWYgKHBlbmFsdHkgPCBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eSkKICAgICAgICAgICAgICAgIHBlbmFsdHkgPSBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eTsKICAgICAgICAgICAgb3V0U2NvcmUgKz0gcGVuYWx0eTsKCiAgICAgICAgICAgIC8vIEFwcGx5IHVubWF0Y2hlZCBwZW5hbHR5CiAgICAgICAgICAgIGludCB1bm1hdGNoZWQgPSAoaW50KShzdHIgLSBzdHJCZWdpbikgLSBuZXh0TWF0Y2g7CiAgICAgICAgICAgIG91dFNjb3JlICs9IHVubWF0Y2hlZF9sZXR0ZXJfcGVuYWx0eSAqIHVubWF0Y2hlZDsKCiAgICAgICAgICAgIC8vIEFwcGx5IG9yZGVyaW5nIGJvbnVzZXMKICAgICAgICAgICAgZm9yIChpbnQgaSA9IDA7IGkgPCBuZXh0TWF0Y2g7ICsraSkgewogICAgICAgICAgICAgICAgdWludDhfdCBjdXJySWR4ID0gbWF0Y2hlc1tpXTsKCiAgICAgICAgICAgICAgICBpZiAoaSA+IDApIHsKICAgICAgICAgICAgICAgICAgICB1aW50OF90IHByZXZJZHggPSBtYXRjaGVzW2kgLSAxXTsKCiAgICAgICAgICAgICAgICAgICAgLy8gU2VxdWVudGlhbAogICAgICAgICAgICAgICAgICAgIGlmIChjdXJySWR4ID09IChwcmV2SWR4ICsgMSkpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcXVlbnRpYWxfYm9udXM7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQ2hlY2sgZm9yIGJvbnVzZXMgYmFzZWQgb24gbmVpZ2hib3IgY2hhcmFjdGVyIHZhbHVlCiAgICAgICAgICAgICAgICBpZiAoY3VycklkeCA+IDApIHsKICAgICAgICAgICAgICAgICAgICAvLyBDYW1lbCBjYXNlCiAgICAgICAgICAgICAgICAgICAgY2hhciBuZWlnaGJvciA9IHN0ckJlZ2luW2N1cnJJZHggLSAxXTsKICAgICAgICAgICAgICAgICAgICBjaGFyIGN1cnIgPSBzdHJCZWdpbltjdXJySWR4XTsKICAgICAgICAgICAgICAgICAgICBpZiAoOjppc2xvd2VyKG5laWdoYm9yKSAmJiA6OmlzdXBwZXIoY3VycikpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGNhbWVsX2JvbnVzOwoKICAgICAgICAgICAgICAgICAgICAvLyBTZXBhcmF0b3IKICAgICAgICAgICAgICAgICAgICBib29sIG5laWdoYm9yU2VwYXJhdG9yID0gbmVpZ2hib3IgPT0gJ18nIHx8IG5laWdoYm9yID09ICcgJzsKICAgICAgICAgICAgICAgICAgICBpZiAobmVpZ2hib3JTZXBhcmF0b3IpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcGFyYXRvcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGVsc2UgewogICAgICAgICAgICAgICAgICAgIC8vIEZpcnN0IGxldHRlcgogICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGZpcnN0X2xldHRlcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgfQogICAgICAgIH0KCiAgICAgICAgLy8gUmV0dXJuIGJlc3QgcmVzdWx0CiAgICAgICAgaWYgKHJlY3Vyc2l2ZU1hdGNoICYmICghbWF0Y2hlZCB8fCBiZXN0UmVjdXJzaXZlU2NvcmUgPiBvdXRTY29yZSkpIHsKICAgICAgICAgICAgLy8gUmVjdXJzaXZlIHNjb3JlIGlzIGJldHRlciB0aGFuICJ0aGlzIgogICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIG1heE1hdGNoZXMpOwogICAgICAgICAgICBvdXRTY29yZSA9IGJlc3RSZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgcmV0dXJuIHRydWU7CiAgICAgICAgfQogICAgICAgIGVsc2UgaWYgKG1hdGNoZWQpIHsKICAgICAgICAgICAgLy8gInRoaXMiIHNjb3JlIGlzIGJldHRlciB0aGFuIHJlY3Vyc2l2ZQogICAgICAgICAgICByZXR1cm4gdHJ1ZTsKICAgICAgICB9CiAgICAgICAgZWxzZSB7CiAgICAgICAgICAgIC8vIG5vIG1hdGNoCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKICAgICAgICB9CiAgICB9Cn0gLy8gbmFtZXNwYWNlIGZ0cwoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9JTVBMRU1FTlRBVElPTgoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9ICg==",
 "tests/logic_test.cpp": "Ly8gU3RhbmRhbG9uZSBsb2dpYyB0ZXN0IGZvciB0aGUgc29uZy1zZWFyY2ggcG9ydC4KLy8gTWlycm9ycyB0aGUgZnV6enkgc2VhcmNoICsgc29ydCArIHNjcm9sbC13aW5kb3cgbG9naWMgaW4gc3JjL21haW4uY3BwCi8vIHdpdGhvdXQgR2VvZGUgaGVhZGVycywgc28gaXQgY2FuIHJ1biBvbiBwbGFpbiBHQ0MuCiNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OCiNpbmNsdWRlICJmdHNfZnV6enlfbWF0Y2guaCIKCiNpbmNsdWRlIDxhbGdvcml0aG0+CiNpbmNsdWRlIDxjYXNzZXJ0PgojaW5jbHVkZSA8Y21hdGg+CiNpbmNsdWRlIDxpb3N0cmVhbT4KI2luY2x1ZGUgPHNzdHJlYW0+CiNpbmNsdWRlIDxzdHJpbmc+CiNpbmNsdWRlIDx2ZWN0b3I+CgpzdHJ1Y3QgRmFrZVNvbmcgeyBpbnQgaWQ7IHN0ZDo6c3RyaW5nIG5hbWU7IHN0ZDo6c3RyaW5nIGFydGlzdDsgfTsKCnN0YXRpYyBzdGQ6OnZlY3RvcjxzdGQ6OnBhaXI8aW50LCBGYWtlU29uZz4+IGRvX3NlYXJjaCgKICAgIGNvbnN0IHN0ZDo6dmVjdG9yPHN0ZDo6cGFpcjxzdGQ6OnN0cmluZywgRmFrZVNvbmc+PiYgZG93bmxvYWRlZCwKICAgIGNvbnN0IHN0ZDo6c3RyaW5nJiBpbnB1dCkgewogICAgc3RkOjp2ZWN0b3I8c3RkOjpwYWlyPGludCwgRmFrZVNvbmc+PiBzb25nczsKICAgIHNvbmdzLnJlc2VydmUoZG93bmxvYWRlZC5zaXplKCkpOwogICAgY29uc3QgYm9vbCBlbXB0eV9pbnB1dCA9IGlucHV0LmVtcHR5KCk7CiAgICBmb3IgKGNvbnN0IGF1dG8mIFtpbmRleF9zdHIsIHNvbmddIDogZG93bmxvYWRlZCkgewogICAgICAgIGludCBzY29yZSA9IDA7CiAgICAgICAgaWYgKCFlbXB0eV9pbnB1dCAmJiAhZnRzOjpmdXp6eV9tYXRjaChpbnB1dC5jX3N0cigpLCBpbmRleF9zdHIuY19zdHIoKSwgc2NvcmUpKSBjb250aW51ZTsKICAgICAgICBzb25ncy5wdXNoX2JhY2soe3Njb3JlLCBzb25nfSk7CiAgICB9CiAgICBpZiAoIWVtcHR5X2lucHV0ICYmIHNvbmdzLnNpemUoKSA+IDEpIHsKICAgICAgICBzdGQ6OnNvcnQoc29uZ3MuYmVnaW4oKSwgc29uZ3MuZW5kKCksIFsmXShjb25zdCBhdXRvJiBhLCBjb25zdCBhdXRvJiBiKSB7CiAgICAgICAgICAgIHJldHVybiBhLmZpcnN0ID4gYi5maXJzdDsKICAgICAgICB9KTsKICAgIH0KICAgIHJldHVybiBzb25nczsKfQoKaW50IG1haW4oKSB7CiAgICBzdGQ6OnZlY3RvcjxzdGQ6OnBhaXI8c3RkOjpzdHJpbmcsIEZha2VTb25nPj4gZG93bmxvYWRlZCA9IHsKICAgICAgICB7IlN0ZXJlbyBNYWRuZXNzIEZvcmV2ZXIiLCB7MSwgIlN0ZXJlbyBNYWRuZXNzIiwgIkZvcmV2ZXIifX0sCiAgICAgICAgeyJCYWNrIG9uIFRyYWNrIERKVkkiLCB7MiwgIkJhY2sgb24gVHJhY2siLCAiREpWSSJ9fSwKICAgICAgICB7IlBvbGFyZ2Vpc3QgU3RlcCIsIHszLCAiUG9sYXJnZWlzdCIsICJTdGVwIn19LAogICAgICAgIHsiRHJ5IE91dCBESlZJIiwgezQsICJEcnkgT3V0IiwgIkRKVkkifX0sCiAgICAgICAgeyJCYXNlIEFmdGVyIEJhc2UgREpWSSIsIHs1LCAiQmFzZSBBZnRlciBCYXNlIiwgIkRKVkkifX0sCiAgICB9OwoKICAgIC8vIDEuIGVtcHR5IGlucHV0IHJldHVybnMgZXZlcnl0aGluZywgdW5zb3J0ZWQKICAgIGF1dG8gYWxsID0gZG9fc2VhcmNoKGRvd25sb2FkZWQsICIiKTsKICAgIGFzc2VydChhbGwuc2l6ZSgpID09IDUpOwoKICAgIC8vIDIuIGV4YWN0LWlzaCBxdWVyeSByYW5rcyB0aGUgcmlnaHQgc29uZyBmaXJzdAogICAgYXV0byBkanZpID0gZG9fc2VhcmNoKGRvd25sb2FkZWQsICJkanZpIik7CiAgICBhc3NlcnQoZGp2aS5zaXplKCkgPT0gMyk7CiAgICBmb3IgKGF1dG8mIFtzY29yZSwgc10gOiBkanZpKSAodm9pZClzY29yZSwgYXNzZXJ0KHMuYXJ0aXN0ID09ICJESlZJIik7CgogICAgLy8gMy4gZnV6enkgcXVlcnkgbWF0Y2hlcyBhY3Jvc3MgbmFtZSthcnRpc3QgYmxvYgogICAgYXV0byBwb2xhciA9IGRvX3NlYXJjaChkb3dubG9hZGVkLCAicG9sciIpOwogICAgYXNzZXJ0KHBvbGFyLnNpemUoKSA9PSAxICYmIHBvbGFyWzBdLnNlY29uZC5pZCA9PSAzKTsKCiAgICAvLyA0LiBubyBtYXRjaCAtPiBlbXB0eQogICAgYXNzZXJ0KGRvX3NlYXJjaChkb3dubG9hZGVkLCAienp6cXFxIikuZW1wdHkoKSk7CgogICAgLy8gNS4gc2Nyb2xsIGNsYW1wOiBtaXJyb3JzIHVwZGF0ZV9zY3JvbGxfZWxlbWVudHMgY2xhbXBpbmcKICAgIGNvbnN0ZXhwciBmbG9hdCBoZWlnaHQgPSA1NS5mOyAvLyB3aWRnZXRfaGVpZ2h0ICsgc3BhY2luZwogICAgY29uc3RleHByIGludCBuX3dpZGdldHMgPSA2OwogICAgYXV0byBjbGFtcF9zY3JvbGwgPSBbJl0oc2l6ZV90IGNvdW50LCBmbG9hdCB5KSB7CiAgICAgICAgY29uc3QgYXV0byBtID0gaGVpZ2h0ICogKHN0YXRpY19jYXN0PGludD4oY291bnQpIC0gbl93aWRnZXRzICsgMyk7CiAgICAgICAgaWYgKG0gPCAwKSByZXR1cm4gMC5mOwogICAgICAgIHJldHVybiBzdGQ6OmNsYW1wKHksIDAuZiwgbSk7CiAgICB9OwogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgwLCA5OTkuZikgPT0gMC5mKTsgICAvLyBlbXB0eSBsaXN0IHBpbnMgdG8gMAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgzLCA5OTkuZikgPT0gMC5mKTsgICAvLyBzaG9ydCBsaXN0IHBpbnMgdG8gMAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgyMCwgLTUwLmYpID09IDAuZik7ICAvLyBuZWdhdGl2ZSBwaW5zIHRvIDAKICAgIGFzc2VydChjbGFtcF9zY3JvbGwoMjAsIDFlOWYpID09IGhlaWdodCAqICgyMCAtIG5fd2lkZ2V0cyArIDMpKTsgLy8gY2FwcyBhdCBtYXgKICAgIGFzc2VydChjbGFtcF9zY3JvbGwoMjAsIDEwMC5mKSA9PSAxMDAuZik7IC8vIGluLXJhbmdlIHVudG91Y2hlZAoKICAgIC8vIDYuIHJlc3VsdCBsYWJlbCBwbHVyYWxpemF0aW9uCiAgICBhdXRvIGxhYmVsID0gW10oc2l6ZV90IG4pIHsKICAgICAgICBzdGQ6OnN0cmluZ3N0cmVhbSBzOyBzIDw8IG4gPDwgIiByZXN1bHQiOyBpZiAobiAhPSAxKSBzIDw8ICJzIjsgcmV0dXJuIHMuc3RyKCk7CiAgICB9OwogICAgYXNzZXJ0KGxhYmVsKDApID09ICIwIHJlc3VsdHMiICYmIGxhYmVsKDEpID09ICIxIHJlc3VsdCIgJiYgbGFiZWwoNSkgPT0gIjUgcmVzdWx0cyIpOwoKICAgIHN0ZDo6Y291dCA8PCAiQUxMIExPR0lDIFRFU1RTIFBBU1NFRCIgPDwgc3RkOjplbmRsOwogICAgcmV0dXJuIDA7Cn0K"
}
ROOT = pathlib.Path('/kaggle/working/song-search')
for rel, b64data in FILES.items():
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64data))
    print('wrote', p, p.stat().st_size, 'bytes')

# ---- version alignment: nothing below is ever hardcoded ----
# geode: the SDK compares major.minor of mod.json["geode"] against the SDK's
# own VERSION file, with a (buggy) regex that keeps any leading "v" intact,
# so "v5.11.1" reduces to "v5.11" vs "5.11" and FATAL_ERRORs. Read the
# real SDK version and strip the prefix if it is there.
sdk_ver = pathlib.Path('/tmp/work/sdk/VERSION').read_text().strip().lstrip('v')
mj = ROOT / 'mod.json'
m = json.loads(mj.read_text())
if m.get('geode') != sdk_ver:
    print('mod.json geode', m.get('geode'), '->', sdk_ver)
    m['geode'] = sdk_ver
    mj.write_text(json.dumps(m, indent=2) + chr(10))
print('mod.json geode =', m['geode'])

# gd.win: the version actually present in the bindings clone we compile
# against (written by cell 3 from the bindings/ listing, cross-checked with
# the SDK's own GEODE_GD_VERSION block). The literal in the embedded mod.json
# above is only a placeholder and is always overwritten here.
gd_win = pathlib.Path('/tmp/work/gd-win.txt').read_text().strip().lstrip('v')
if not gd_win:
    # fallback: parse the SDK's declared GD version
    txt = pathlib.Path('/tmp/work/sdk/CMakeLists.txt').read_text()
    block = txt.split('# begin GEODE_GD_VERSION', 1)[1].split('# end GEODE_GD_VERSION', 1)[0]
    for line in block.splitlines():
        if line.strip().startswith('#') and 'win:' in line:
            gd_win = line.split('win:', 1)[1].strip()
            break
assert gd_win, 'gd.win could not be discovered'
m.setdefault('gd', {})['win'] = gd_win
for plat in ('android', 'mac', 'ios'):
    m['gd'].setdefault(plat, '*')
mj.write_text(json.dumps(m, indent=2) + chr(10))
print('mod.json gd.win =', m['gd']['win'], '(discovered from bindings clone)')

In [ ]:
%%bash

set -eo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "4. mod source + native logic test"
cd /kaggle/working/song-search || exit 10
find . -type f | sort | tee -a "$LOG"
g++ -std=c++20 -O2 -Wall -I src -o /tmp/work/logic_test tests/logic_test.cpp
/tmp/work/logic_test | tee -a "$LOG"
log "logic test rc=$?"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "5. configure (cross-compile -> Win64)"
cd /kaggle/working/song-search || exit 11
export PATH=/usr/local/bin:/tmp/work/geode-cli:/tmp/work/xwin:$PATH
export SPLAT_DIR=/tmp/work/splat
export TOOLCHAIN=/tmp/work/toolchain/clang-cl-msvc.cmake
export HOST_ARCH=x86_64
export GEODE_SDK=/tmp/work/sdk
export GEODE_BINDINGS_REPO_PATH=/tmp/work/bindings

rm -rf build
cmake \
  -DCMAKE_TOOLCHAIN_FILE=$TOOLCHAIN \
  -DCMAKE_BUILD_TYPE=Release \
  -DGEODE_TARGET_PLATFORM=Win64 \
  -G Ninja \
  -B build > /tmp/work/configure.log 2>&1
RC=$?
tail -50 /tmp/work/configure.log | tee -a "$LOG"
echo "configure rc=$RC" | tee -a "$LOG"
[ $RC -eq 0 ] || exit 12

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "6. build"
cd /kaggle/working/song-search || exit 13
export PATH=/usr/local/bin:/tmp/work/geode-cli:$PATH
export GEODE_SDK=/tmp/work/sdk
time cmake --build build --config Release > /tmp/work/build.log 2>&1
RC=$?
tail -70 /tmp/work/build.log | tee -a "$LOG"
echo "build rc=$RC" | tee -a "$LOG"
[ $RC -eq 0 ] || exit 14
log "BUILD DONE"
find build -maxdepth 3 -type f \( -name '*.geode' -o -name '*.dll' -o -name '*.lib' \) \
  -exec ls -la {} \; | tee -a "$LOG"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
cd /kaggle/working/song-search || exit 15
GEO=$(find build -maxdepth 3 -name '*.geode' 2>/dev/null | head -1)
if [ -z "$GEO" ]; then
  echo "NO .geode PRODUCED" | tee -a "$LOG"
  find build -maxdepth 3 -type f 2>/dev/null | head -30 | tee -a "$LOG"
  exit 16
fi
echo "geode: $GEO ($(stat -c%s "$GEO") bytes)" | tee -a "$LOG"
unzip -l "$GEO" | tee -a "$LOG"

log "7. DLL architecture check"
mkdir -p /tmp/work/geocheck
cd /tmp/work/geocheck
rm -rf ./* 2>/dev/null || true
unzip -o -q "/kaggle/working/song-search/$GEO"
DLL=$(find . -name '*.dll' | head -1)
echo "mod dll: $DLL" | tee -a "$LOG"
python3 - "$DLL" <<'PY' | tee -a /kaggle/working/BUILD-LOG.txt
import sys, struct
def pe_info(p):
    d = open(p,'rb').read()
    assert d[:2]==b'MZ'
    e = struct.unpack_from('<I', d, 0x3c)[0]
    assert d[e:e+4]==b'PE\x00\x00'
    mach  = struct.unpack_from('<H', d, e+4)[0]
    chars = struct.unpack_from('<H', d, e+22)[0]
    magic = struct.unpack_from('<H', d, e+24)[0]
    return dict(
      machine={0x14c:'i386 (32-bit)',0x8664:'x86_64 (64-bit)',0xaa64:'arm64'}.get(mach,hex(mach)),
      is_dll=bool(chars & 0x2000),
      pe_format='PE32+' if magic==0x20b else 'PE32',
      size=len(d))
i = pe_info(sys.argv[1])
for k,v in i.items(): print('  %s: %s' % (k,v))
ok = i['machine'].startswith('x86_64') and i['is_dll'] and i['pe_format']=='PE32+'
gd_win = pathlib.Path('/tmp/work/gd-win.txt').read_text().strip()
print('  ARCH MATCHES GD %s (x86_64):' % gd_win, 'YES' if ok else 'NO')
assert ok, 'built DLL is not x86_64 PE32+'
PY

log "8. mod.json sanity"
python3 - <<'PY' | tee -a /kaggle/working/BUILD-LOG.txt
import json, pathlib
m = json.load(open('/kaggle/working/song-search/mod.json'))
gd_win = pathlib.Path('/tmp/work/gd-win.txt').read_text().strip()
print('id     :', m['id'])
print('version:', m['version'])
print('geode  :', m['geode'])
print('gd.win :', m['gd']['win'])
# compare against the version discovered from the bindings clone, never a literal
assert m['gd']['win'] == gd_win, (m['gd']['win'], gd_win)
print('mod.json OK')
PY

log "ALL DONE"